<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L52_Production_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L52: Production Deployment - Serving, API, and Scaling

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 52 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Expose an LLM as an HTTP API with FastAPI
- Add request validation, timeouts, and basic error handling
- Understand scaling options: batching, multiple workers, GPU

---

## Concept: Production Serving

**Serving**: load model once, handle many requests. **API**: REST or gRPC; input (prompt, params), output (generated text). **Scaling**: vertical (bigger GPU), horizontal (more replicas), batching (dynamic batch). We build a minimal FastAPI endpoint.

---

In [ ]:
!pip install fastapi uvicorn transformers torch -q
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: FastAPI App and Model

---

In [ ]:
app = FastAPI(title="LLM API")
generator = pipeline("text-generation", model="gpt2", device=0 if torch.cuda.is_available() else -1)

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 50

class GenerateResponse(BaseModel):
    text: str

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    if len(req.prompt) > 2000:
        raise HTTPException(status_code=400, detail="Prompt too long")
    out = generator(req.prompt, max_new_tokens=min(req.max_new_tokens, 100), do_sample=False, pad_token_id=generator.tokenizer.eos_token_id)
    text = out[0]["generated_text"][len(req.prompt):].strip()
    return GenerateResponse(text=text)

print("Run with: uvicorn script:app --host 0.0.0.0 --port 8000")

## Part 2: Health and Readiness

---

In [ ]:
@app.get("/health")
def health():
    return {"status": "ok"}

# Fix typo: @ app -> @app
print("Add /health for load balancers; /ready after model is loaded.")

## Part 3: Scaling and Monitoring

---

In [ ]:
# Fix: remove space in @ app above - in real code use @app.get("/health")
print("Scaling: run multiple uvicorn workers or use TorchServe / TGI / vLLM for batching.")
print("Monitoring: log latency, token/s, errors; use Prometheus + Grafana.")

## Exercises

1. Add a /ready endpoint that checks model is loaded.
2. Add request_id and log it; measure P95 latency.
3. Run 2 workers with gunicorn and a small load test.

---

## Key Takeaways

1. FastAPI + pipeline = minimal LLM API; add validation and limits.
2. Health/ready endpoints and timeouts are important for production.
3. For high throughput use dedicated serving (vLLM, TGI) with continuous batching.

---

## Next Lesson

**L53: Cost Optimization** – Inference cost and resource management.

---